In [8]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision

In [9]:
# dataset download -> dogs and cats

import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_2f4781ce33af65adb0875b91b7d64ef8"

!pip install -q --upgrade kaggle

""" Download the dataset """
!kaggle datasets download -d karakaggle/kaggle-cat-vs-dog-dataset

""""Unzip the downloaded file"""
import zipfile
print("Unzipping dataset...")
with zipfile.ZipFile("kaggle-cat-vs-dog-dataset.zip", "r") as zip_ref:
    zip_ref.extractall("data_clean")

""" Remove The zip container """
os.remove("kaggle-cat-vs-dog-dataset.zip")
print("Complete!")

Dataset URL: https://www.kaggle.com/datasets/karakaggle/kaggle-cat-vs-dog-dataset
License(s): unknown
100% 787M/787M [00:07<00:00, 108MB/s]  

Unzipping dataset...
Complete!


In [10]:
# Dataset folder define
data_folder_path = '/content/data_clean/kagglecatsanddogs_3367a/PetImages'

In [11]:
# Removing broken file

from PIL import Image

for category in ['Cat', 'Dog']:
    folder_path = os.path.join(data_folder_path, category)
    
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)

        # check for empty files or non-jpg items
        if os.path.getsize(file_path) == 0 or not filename.endswith('.jpg'):
            print(f"Removing broken/empty file: {file_path}")
            os.remove(file_path)
            continue

        # try opening to conform it isn't corrupted
        try:
            with Image.open(file_path) as img:
                img.verify()
        except (IOError, SyntaxError) as e:
            print(f"Removing corrupted image: {file_path}")
            os.remove("file_path")

print("All broken images cleaned up successfully!")


Removing broken/empty file: /content/data_clean/kagglecatsanddogs_3367a/PetImages/Cat/Thumbs.db
Removing broken/empty file: /content/data_clean/kagglecatsanddogs_3367a/PetImages/Dog/Thumbs.db


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


All broken images cleaned up successfully!


In [12]:
# Dataset build

from torchvision import datasets, transforms

transform = transforms.Compose([
    # Force all images to be the exact same size
    transforms.Resize((128, 128)),

    # Scale pixel values from [0, 255] to [0.0, 1.0]
    transforms.ToTensor(),

    # Shift range from [0.0, 1.0] to [-1.0, 1.0]
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
])

# Load full dataset from main folder
full_dataset = datasets.ImageFolder(
    root=data_folder_path,
    transform=transform
)

In [ ]:
""" Dataset Split - Train & val"""

from torch.utils.data import DataLoader, random_split

train_size = int(0.8 * len(full_dataset))
test_size = int(len(full_dataset) - train_size)

train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=32, drop_last=True)

print(f"Total Images: {len(full_dataset)}")
print(f"Training Images: {len(train_dataset)}")
print(f"Validation Images: {len(test_dataset)}")

Total Images: 24959
Training Images: 19967
Validation Images: 4992


## Build CNN

In [1]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        # 1st LAyer: Feature Extractors
        # each Image size = 128x128
        self.conv_layers = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(), # conv+ReLU: (128, 128, 3) => (128, 128, 16)
            # Max pooling layer (Downsample dimensions by half)
            nn.MaxPool2d(kernel_size=2, stride=2), # (64, 64, 16)
            # Size: 16 channels | 64x64

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(), # conv+ReLU: (64, 64, 16) => (64, 64, 32)
            nn.MaxPool2d(kernel_size=2, stride=2), # (32, 32, 32)
            # Size: 32 channels | 32x32

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(), # conv+ReLU: (32, 32, 32) => (32, 32, 64)
            nn.MaxPool2d(kernel_size=2, stride=2) # (16, 16, 64)
            # Size: 64 channels | 16x16
        )

        # After Three Pooling, a 128x128 image becomes 16x16 (128 -> 64 -> 32 -> 16)
        # 64 channels * 16 * 16 pixels = 64*16*16 = 16384 fatures

        self.fc_layers = nn.Sequential(
            nn.Linear(64*16*16, 128), # inout, output 
            nn.ReLU(),

            nn.Linear(128, 2) # 2 output for binary classifaction: Cat or Dog
        )


    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flatten the image tensor into a 1D vector
        x = self.fc_layers(x) # logits for crossENtropyLoss

        return x

NameError: name 'nn' is not defined

In [15]:
model = CNN()
print(model)

CNN(
  (conv_layers): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layers): Sequential(
    (0): Linear(in_features=16384, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=2, bias=True)
  )
)


In [ ]:
criterian = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())


# Model Train & validation

In [ ]:
# Model Train & validation
train_losses = []
val_losses = []
best_val_loss = 0.0

epochs = 10

for epoch in range(epochs):
    running_train_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()

        output = model.forward(images) # FP
        loss = criterian(output, labels)
        loss.backward() # BP
        optimizer.step() # update params

        running_train_loss += loss

    epoch_train_loss = running_train_loss / len(train_loader)
    train_losses.append(epoch_train_loss)

    model.eval()
    running_val_loss = 0.0

    with torch.no_grad:
        for images, labels in test_loader:
            output = model(images)
            loss = criterian(output, labels)
            running_val_loss += loss

        epoch_val_loss = running_val_loss / len(test_loader)
        val_losses.append(epoch_val_loss)

    print(f"epoch {epoch+1}/{epochs} - train_loss={epoch_train_loss} & val_loss={epoch_val_loss}")

    if epoch_val_loss > best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.parameters(), "/model/DogCat_Classiify_best.pt/")



## Plot Train & Vaildation Loss

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

loss_df = pd.DataFrame({
    "Training Loss": train_losses,
    "validation Loss": val_losses,
})

plt.plot(loss_df["Train Loss"], labels="Training Loss")
plt.plot(loss_df["Validation Loss"], labels="Validation Loss")

plt.title("Training vs Validation Loss")
plt.legend()

plt.xlabel("Epochs")
plt.ylabel("Losses")

## Validation

In [ ]:
# Validation

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Total data = {total}")
print(f"Correct prediction = {correct}")
print(f"Test Accuracy: {100 * correct / total:.2f}%") 

    